# SHYPS subsystem memory

**[DEMO]** — packaged `MemoryExperiment` and `SHYPSCode`.

The patch declares the stabilizer centre and gauge generators once. LightStim derives detectors from the X/Z gauge-measurement history.

This notebook retains the two-round SHYPS X/Z integration and visualization checks: r=3 has 49 data qubits, 9 protected logicals, code distance 4, and 98 readout ancillas. These checks do not establish circuit distance or reproduce paper performance.

The weight-2 Bacon–Shor demo now lives in [memory_bacon_shor.ipynb](memory_bacon_shor.ipynb), including dedicated SE and CPU MWPM results.

In [1]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.shyps import SHYPSCode

## Ideal memory checks

Each returned observable belongs to a protected logical qubit. Gauge-state directions are excluded from that count. The small ideal samples below must contain no detector or observable flips, and the detector error model must be constructible.

In [2]:
def check_ideal_memory(circuit, expected_logicals):
    detectors, observables = circuit.compile_detector_sampler(seed=71).sample(
        128, separate_observables=True
    )
    assert circuit.num_observables == expected_logicals
    assert observables.shape == (128, expected_logicals)
    assert not detectors.any()
    assert not observables.any()
    assert circuit.detector_error_model().num_observables == expected_logicals
    return {
        "physical_qubits": circuit.num_qubits,
        "detectors": circuit.num_detectors,
        "protected_observables": circuit.num_observables,
        "ideal_detector_flips": int(detectors.sum()),
        "ideal_observable_flips": int(observables.sum()),
    }

In [3]:
shyps_z = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="Z"
).build()
shyps_x = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="X"
).build()
print("SHYPS Z:", check_ideal_memory(shyps_z, 9))
print("SHYPS X:", check_ideal_memory(shyps_x, 9))

SHYPS Z: {'physical_qubits': 147, 'detectors': 148, 'protected_observables': 9, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}
SHYPS X: {'physical_qubits': 147, 'detectors': 148, 'protected_observables': 9, 'ideal_detector_flips': 0, 'ideal_observable_flips': 0}


## SHYPS visualization (r=3)

These views use the packaged SHYPS patch and the `shyps_z` circuit built above by
LightStim's `MemoryExperiment`. Detectors are the existing Builder/Tracker output.

The first figure shows the actual stored display coordinates and one X/Z gauge
example. Black circles are data; red/blue squares are X/Z readout ancillas.
The layout has 49 data qubits and 98 readout ancillas. Long lines represent the
declared connectivity; these display coordinates do not impose nearest-neighbor gates.

The second view is the same **detector slices with operations** used for Bacon-Shor.
Both views are also exported under `playground/subsystem_visualization/`.


In [4]:
from io import StringIO
import matplotlib.pyplot as plt
from IPython.display import SVG, display

visualization_dir = ROOT / "playground" / "subsystem_visualization"
visualization_dir.mkdir(parents=True, exist_ok=True)
shyps_patch_view = SHYPSCode(r=3)
coords = shyps_patch_view.qubit_coords
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6), layout="constrained")
roles = [
    (shyps_patch_view.data_indices, "Data (49)", "#202938", "o"),
    (shyps_patch_view.syndrome_indices_x, "X ancillas (49)", "#e44545", "s"),
    (shyps_patch_view.syndrome_indices_z, "Z ancillas (49)", "#2783d4", "s"),
]
for ax, basis, color in zip(axes, ("X", "Z"), ("#e44545", "#2783d4")):
    for indices, label, role_color, marker in roles:
        ordered = sorted(indices)
        ax.scatter(*zip(*(coords[q] for q in ordered)), c=role_color,
                   marker=marker, s=22, alpha=0.38, label=label, zorder=2)
    gauge = next(g for g in shyps_patch_view.gauges if g["type"] == basis)
    ancilla = gauge["syn_idx"]
    for data in gauge["data_indices"]:
        ax.plot(*zip(coords[ancilla], coords[data]), color=color, lw=1.8, zorder=1)
    selected = [ancilla] + gauge["data_indices"]
    for q in selected:
        ax.scatter(*coords[q], c=color if q == ancilla else "#202938",
                   marker="s" if q == ancilla else "o", s=45, zorder=3)
        ax.annotate(f"q{q}", coords[q], xytext=(5, -10),
                    textcoords="offset points", fontsize=8)
    pauli = " ".join(f"{basis}{q}" for q in gauge["data_indices"])
    ax.set(title=f"{basis} gauge example: {pauli} (ancilla q{ancilla})",
           xlabel="Display x", ylabel="Display y", xlim=(-1, 16), ylim=(16, -1))
    ax.set_aspect("equal")
    ax.legend(loc="lower right", framealpha=0.95, fontsize=8)
fig.suptitle("SHYPS r=3: declared patch layout and representative gauge connections")
layout_svg = StringIO()
fig.savefig(layout_svg, format="svg")
(visualization_dir / "shyps_r3_layout.svg").write_text(layout_svg.getvalue())
fig.savefig(visualization_dir / "shyps_r3_layout.png", dpi=160)
plt.close(fig)
display(SVG(layout_svg.getvalue()))


In [5]:
# Full two-round SHYPS memory view.
# Switch to shyps_x for X memory. Add tick=range(start, stop) to zoom in time.
shyps_detector_view = shyps_z.diagram("detslice-with-ops-svg")
(visualization_dir / "shyps_r3_detslice.svg").write_text(str(shyps_detector_view))
display(shyps_detector_view)


## Circuit noise and detector error models

The same public protocol accepts a `NoiseConfig`. Here we add two-qubit and readout noise, then construct each DEM without requesting graphlike decomposition. Sampling or constructing a DEM is not decoding; these counts do not establish a logical error rate, threshold or single-shot performance.

In [6]:
noise = NoiseConfig(p_2q=0.001, p_meas=0.001)

shyps_noisy = MemoryExperiment(
    qec_patch=SHYPSCode(r=3), rounds=2, basis="Z",
    noise_params=noise,
).build()

shyps_dem = shyps_noisy.detector_error_model()
assert shyps_dem.num_errors > 0 and shyps_dem.num_observables == 9
print("SHYPS DEM:", {"error_terms": shyps_dem.num_errors, "observables": shyps_dem.num_observables})

SHYPS DEM: {'error_terms': 2009, 'observables': 9}


## Construction references

- [Malcolm et al., *Computing Efficiently in QLDPC Codes*, §§VIII.4–VIII.5](https://arxiv.org/html/2502.07150v2): the SHYPS matrices, parameters and paired bare logical representatives.
- [SHYPS implementation notes](../../lightstim/qec_code/shyps/README.md): supported sizes, indexing and ancilla allocation.

The SHYPS default is a generic finite gauge-extraction schedule. It does not reproduce the paper's logical Clifford compiler, decoder or optimized performance results.